# Análise de Dados - Titanic

**Primeiro projeto usando Pandas**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_palette("viridis")
pd.set_option('display.max_columns', None)

print("Bibliotecas carregadas!")

In [ ]:
df = pd.read_csv('titanic.csv')

print(f"Dataset carregado: {df.shape[0]} passageiros e {df.shape[1]} colunas")
df.head()

In [ ]:
df_clean = df.copy()

# Tirei a coluna Cabin porque quase não tinha informação
df_clean = df_clean.drop('Cabin', axis=1)

# Preenchi a idade usando a mediana de cada classe
df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Só tinha 2 faltando no porto de embarque
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

print("Dados depois da limpeza:")
print(df_clean.isnull().sum())

In [ ]:
# Peguei o título das pessoas (Mr, Miss, Mrs...)
df_clean['Title'] = df_clean['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Juntei títulos raros
df_clean['Title'] = df_clean['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 
                                               'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df_clean['Title'] = df_clean['Title'].replace(['Mlle', 'Ms'], 'Miss')
df_clean['Title'] = df_clean['Title'].replace('Mme', 'Mrs')

# Criei coluna de tamanho da família
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

print(df_clean['Title'].value_counts())

In [ ]:
# Gráficos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df_clean, x='Survived', ax=axes[0,0])
axes[0,0].set_title('Quantos sobreviveram?')

sns.countplot(data=df_clean, x='Sex', hue='Survived', ax=axes[0,1])
axes[0,1].set_title('Sobrevivência por Sexo')

sns.countplot(data=df_clean, x='Pclass', hue='Survived', ax=axes[1,0])
axes[1,0].set_title('Sobrevivência por Classe')

sns.histplot(data=df_clean, x='Age', hue='Survived', kde=True, ax=axes[1,1])
axes[1,1].set_title('Sobrevivência por Idade')

plt.tight_layout()
plt.show()

In [ ]:
print("Taxa geral de sobrevivência:", round(df_clean['Survived'].mean()*100, 1), "%")
print("\nSobrevivência por sexo:")
print(df_clean.groupby('Sex')['Survived'].mean())
print("\nSobrevivência por classe:")
print(df_clean.groupby('Pclass')['Survived'].mean())